In [1]:
import RNA

# ================= 1. 输入待检测引物序列 =================
# 课题确定的 RPA Forward Primer (5' -> 3')
fwd_primer = "ACGACATCGATTTATAGCACCATCTGAAATCGGT"
rev_primer = "GTCGTATCCAGTGCAGGGTCCGAG"

# 判定阈值 (kcal/mol)
DG_THRESHOLD = -2.0


# ================= 2. 定义引物自身发卡自由能计算函数 =================
def check_primer_hairpin(primer_seq: str, primer_name: str = "Primer"):
    """使用 ViennaRNA (RNAfold) 评估引物自身发卡折叠自由能 MFE

    参数:
    ----
    primer_seq : str
        引物 DNA 序列 (5' -> 3')
    primer_name : str
        引物名称
    """
    # 转换为大写并处理空白
    seq = primer_seq.upper().strip()

    # 创建 fold_compound 对象 (ViennaRNA 支持计算单链核酸折叠)
    fc = RNA.fold_compound(seq)
    structure, mfe = fc.mfe()

    # 判断是否满足 ΔG > -2.0 kcal/mol 要求
    is_passed = mfe > DG_THRESHOLD

    print(f"=== {primer_name} 自身发卡与二级结构分析 ===")
    print(f"序列 (5'->3') : {seq}")
    print(f"序列长度      : {len(seq)} nt")
    print(f"预测二级结构  : {structure}")
    print(f"自身折叠 MFE  : {mfe:.2f} kcal/mol")
    print(f"阈值要求      : ΔG > {DG_THRESHOLD:.1f} kcal/mol")
    print(
        f"判定结果      : {'【合格 PASS】(无恶性自身发卡)' if is_passed else '【警告 FAIL】(自身发卡过强，可能阻碍退火扩增)'}"
    )
    print("-" * 50)

    return {"name": primer_name, "structure": structure, "mfe": mfe, "passed": is_passed}


# ================= 3. 运行检测 =================
res_fwd = check_primer_hairpin(fwd_primer, "RPA Forward Primer")
res_rev = check_primer_hairpin(rev_primer, "RPA Reverse Primer")

=== RPA Forward Primer 自身发卡与二级结构分析 ===
序列 (5'->3') : ACGACATCGATTTATAGCACCATCTGAAATCGGT
序列长度      : 34 nt
预测二级结构  : .....((((((((.(((......)))))))))))
自身折叠 MFE  : -6.10 kcal/mol
阈值要求      : ΔG > -2.0 kcal/mol
判定结果      : 【警告 FAIL】(自身发卡过强，可能阻碍退火扩增)
--------------------------------------------------
=== RPA Reverse Primer 自身发卡与二级结构分析 ===
序列 (5'->3') : GTCGTATCCAGTGCAGGGTCCGAG
序列长度      : 24 nt
预测二级结构  : .(((.((((......)))).))).
自身折叠 MFE  : -6.20 kcal/mol
阈值要求      : ΔG > -2.0 kcal/mol
判定结果      : 【警告 FAIL】(自身发卡过强，可能阻碍退火扩增)
--------------------------------------------------
